# Experiment Runner

Orchestration notebook for all experiments.

**Output:** `data/results/<model>/` — one folder per model.
Summaries are generated per-model (same-model pairing).

## Experiments

| # | Task | Calls | Input type |
|---|------|-------|------------|
| 0 | CV Summarisation | 945 | 1 CV each |
| 1 | Pick best candidate | 315 | 3 full CVs + JD |
| 2 | Pick best candidate | 315 | 3 summaries + JD |
| 3 | Leadership potential | 2 × 315 | CV / summary |
| 4 | Competence scoring (isolated) | 2 × 945 | 1 CV / summary each |
| 4b | Competence scoring (comparative) | 2 × 315 | 3 CVs / summaries |
| 5 | Hierarchical job matching | 2 × 315 | CVs / summaries (**Cohere only**) |
| 6 | Pick best — senior JD | 2 × 315 | CVs / summaries |

**Total per model (excl. Exp 5):** ~5,355 API calls.

## Known JD/CV mismatches (7 resume_ids)

These come from the original JobResQA benchmark, not our pipeline.
Both datasets have `jd_cv_mismatch = 1` for these rows — filter during analysis.

In [1]:
import importlib
import sys
import time
from pathlib import Path

import pandas as pd

EXPERIMENTS_DIR = Path.cwd()
if not (EXPERIMENTS_DIR / "runner.py").exists():
    EXPERIMENTS_DIR = Path.cwd() / "experiments"
if str(EXPERIMENTS_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENTS_DIR))

import runner
importlib.reload(runner)
from runner import (
    init_cohere_client, init_client,
    run_experiment, run_per_cv_experiment,
    model_results_dir, DATA_DIR, RESULTS_DIR,
    PROVIDER_COHERE, PROVIDER_OPENROUTER,
)
from constants import CHAT_MODEL

import validate_results
importlib.reload(validate_results)
from validate_results import rerun_bad_rows

import exp0_summarise
import exp1_pick_best_cv
import exp2_pick_best_summary
import exp3_leadership_potential
import exp4_competence_scoring
import exp4b_competence_comparative
import exp5_job_rank
import exp6_senior_jd

df = pd.read_csv(
    DATA_DIR / "final_CV_dataset_experiments_one_row.csv",
    keep_default_na=False,
)
print(f"Dataset: {len(df)} rows")

Dataset: 315 rows


In [2]:
SHARED_SUMMARIES = DATA_DIR / "shared_summaries" / "exp0_summarise_cohere.csv"


def ensure_summaries(model, *, summaries_path=None):
    """Load exp0 summaries for modules that need them (idempotent)."""
    kw = {"summaries_path": str(summaries_path)} if summaries_path else {}
    if exp2_pick_best_summary._loaded_model != model:
        exp2_pick_best_summary.load_summaries(model, **kw)
    if exp6_senior_jd._loaded_model != model:
        exp6_senior_jd.load_summaries(model, **kw)


def run_all(client, model, *, include_exp0=True, include_exp5=False, include_exp6=True):
    """Run the full experiment suite for one model."""
    fix = lambda name: rerun_bad_rows(model, name, client=client)
    t0 = time.time()

    print(f"\n{'=' * 60}")
    print(f"  MODEL: {model}")
    print(f"{'=' * 60}")

    # Exp 0 — Summarise (prerequisite for summary experiments)
    if include_exp0:
        run_per_cv_experiment(df, exp0_summarise.CONFIG, client, model=model)
        fix("exp0_summarise")
        _sum_path = None
    else:
        _sum_path = SHARED_SUMMARIES
        print(f"  Skipping exp0 — using shared Cohere summaries from {_sum_path}")

    # Exp 1 — Pick best (CVs)
    run_experiment(df, exp1_pick_best_cv.CONFIG, client, model=model)
    fix("exp1_pick_best_cv")

    # Exp 2 — Pick best (summaries)
    ensure_summaries(model, summaries_path=_sum_path)
    run_experiment(df, exp2_pick_best_summary.CONFIG, client, model=model)
    fix("exp2_pick_best_summary")

    # Exp 3 — Leadership (CVs + summaries)
    run_experiment(df, exp3_leadership_potential.CONFIG_CV, client, model=model)
    fix("exp3_leadership_cv")
    run_experiment(df, exp3_leadership_potential.CONFIG_SUMMARY, client, model=model)
    fix("exp3_leadership_summary")

    # Exp 4 — Competence scoring, isolated (CVs + summaries)
    run_per_cv_experiment(df, exp4_competence_scoring.CONFIG_CV, client, model=model)
    fix("exp4_competence_cv")
    run_per_cv_experiment(df, exp4_competence_scoring.CONFIG_SUMMARY, client, model=model)
    fix("exp4_competence_summary")

    # Exp 4b — Competence scoring, comparative (CVs + summaries)
    run_experiment(df, exp4b_competence_comparative.CONFIG_CV, client, model=model)
    fix("exp4b_competence_comparative_cv")
    run_experiment(df, exp4b_competence_comparative.CONFIG_SUMMARY, client, model=model)
    fix("exp4b_competence_comparative_summary")

    # Exp 5 — Hierarchical job matching (Cohere only)
    if include_exp5:
        run_experiment(df, exp5_job_rank.CONFIG_CV, client, model=model)
        fix("exp5_job_rank_cv")
        run_experiment(df, exp5_job_rank.CONFIG_SUMMARY, client, model=model)
        fix("exp5_job_rank_summary")

    # Exp 6 — Senior JD (CVs + summaries)
    if include_exp6:
        run_experiment(df, exp6_senior_jd.CONFIG_CV, client, model=model)
        fix("exp6_senior_jd_cv")
        run_experiment(df, exp6_senior_jd.CONFIG_SUMMARY, client, model=model)
        fix("exp6_senior_jd_summary")

    elapsed = time.time() - t0
    print(f"\n  {model} — done in {elapsed/60:.1f} min.\n")

---

## Cohere (direct API)

Runs all experiments including Exp 5 and Exp 6.

In [ ]:
MODEL = CHAT_MODEL
client = init_cohere_client()
print(f"Cohere client ready — model: {MODEL}")

run_all(client, MODEL, include_exp5=True, include_exp6=True)

---

## Multi-model run (OpenRouter)

Runs the full suite (**excluding Exp 0 and Exp 5**) across all OpenRouter models
sequentially. Set `OPENROUTER_API_KEY` in your environment before running.

**Summaries:** All OpenRouter models use the shared Cohere-generated summaries
from `data/shared_summaries/exp0_summarise_cohere.csv`. This controls the
input for summary-based experiments so differences reflect evaluation bias only.

Each model gets its own `data/results/<model>/` folder.
Checkpointing is per-row, so you can interrupt and resume safely.

In [3]:
OPENROUTER_MODELS = [
    "openai/gpt-5",
    "openai/gpt-5-mini",
    "google/gemini-2.5-flash",
    "meta-llama/llama-4-maverick",
    "x-ai/grok-3-mini",
    "deepseek/deepseek-v3.2",
]

or_client = init_client(PROVIDER_OPENROUTER)
print(f"OpenRouter client ready — {len(OPENROUTER_MODELS)} models")
for m in OPENROUTER_MODELS:
    print(f"  • {m}")

OpenRouter client ready — 6 models
  • openai/gpt-5
  • openai/gpt-5-mini
  • google/gemini-2.5-flash
  • meta-llama/llama-4-maverick
  • x-ai/grok-3-mini
  • deepseek/deepseek-v3.2


In [4]:
t_start = time.time()

for i, m in enumerate(OPENROUTER_MODELS, 1):
    print(f"\n>>> Model {i}/{len(OPENROUTER_MODELS)}: {m}")
    run_all(or_client, m, include_exp0=False, include_exp5=False, include_exp6=True)

total = time.time() - t_start
print(f"\nAll models done in {total/60:.1f} min ({total/3600:.1f} h)")


>>> Model 1/6: openai/gpt-5

  MODEL: openai/gpt-5
  Skipping exp0 — using shared Cohere summaries from /Users/tamara/thesis /data/shared_summaries/exp0_summarise_cohere.csv
[exp1_pick_best_cv] Starting experiment (315 rows, 0 already done)...
[exp1_pick_best_cv] Row 1/315 (resume_id=1267, perm=1)...
[exp1_pick_best_cv] Row 2/315 (resume_id=1267, perm=2)...
[exp1_pick_best_cv] Row 3/315 (resume_id=1267, perm=3)...
[exp1_pick_best_cv] Row 4/315 (resume_id=1268, perm=1)...
[exp1_pick_best_cv] Row 5/315 (resume_id=1268, perm=2)...
[exp1_pick_best_cv] Row 6/315 (resume_id=1268, perm=3)...
[exp1_pick_best_cv] Row 7/315 (resume_id=1269, perm=1)...
[exp1_pick_best_cv] Row 8/315 (resume_id=1269, perm=2)...
[exp1_pick_best_cv] Row 9/315 (resume_id=1269, perm=3)...
[exp1_pick_best_cv] Row 10/315 (resume_id=1270, perm=1)...
[exp1_pick_best_cv] Row 11/315 (resume_id=1270, perm=2)...
[exp1_pick_best_cv] Row 12/315 (resume_id=1270, perm=3)...
[exp1_pick_best_cv] Row 13/315 (resume_id=1271, perm=1).

---

## Results summary

In [5]:
import os

print("Result files:")
model_dirs = sorted(
    d for d in RESULTS_DIR.rglob("*")
    if d.is_dir() and any(d.glob("*.csv"))
)
for model_dir in model_dirs:
    csvs = sorted(model_dir.glob("*.csv"))
    total_bytes = sum(os.path.getsize(f) for f in csvs)
    rel = model_dir.relative_to(RESULTS_DIR)
    print(f"\n  [{rel}]  ({len(csvs)} files, {total_bytes/1024:.0f} KB)")
    for f in csvs:
        print(f"    {f.name}")

Result files:

  [command-a-03-2025]  (14 files, 7005 KB)
    FIRST_ATTEMPT_checkpoint_exp0_summarise.csv
    exp0_summarise.csv
    exp1_pick_best_cv.csv
    exp2_pick_best_summary.csv
    exp3_leadership_cv.csv
    exp3_leadership_summary.csv
    exp4_competence_cv.csv
    exp4_competence_summary.csv
    exp4b_competence_comparative_cv.csv
    exp4b_competence_comparative_summary.csv
    exp5_job_rank_cv.csv
    exp5_job_rank_summary.csv
    exp6_senior_jd_cv.csv
    exp6_senior_jd_summary.csv

  [deepseek/deepseek-v3.2]  (10 files, 4259 KB)
    exp1_pick_best_cv.csv
    exp2_pick_best_summary.csv
    exp3_leadership_cv.csv
    exp3_leadership_summary.csv
    exp4_competence_cv.csv
    exp4_competence_summary.csv
    exp4b_competence_comparative_cv.csv
    exp4b_competence_comparative_summary.csv
    exp6_senior_jd_cv.csv
    exp6_senior_jd_summary.csv

  [google/gemini-2.5-flash]  (10 files, 4364 KB)
    exp1_pick_best_cv.csv
    exp2_pick_best_summary.csv
    exp3_leadership_cv.csv